In [1]:
import torch
from PIL import Image
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection
import torchvision.ops as ops

# -----------------------------
# 1️⃣ Config
# -----------------------------
MODEL_ID = "IDEA-Research/grounding-dino-base"
IMAGE_PATH = "../test2.jpeg"
LVIS_LABELS_PATH = "../resources/lvis_labels.txt"

BOX_THRESHOLD = 0.25
TEXT_THRESHOLD = 0.10
CHUNK_SIZE = 50
NMS_IOU_THRESHOLD = 0.5

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# -----------------------------
# 2️⃣ Load model
# -----------------------------
print("Loading model...")

processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForZeroShotObjectDetection.from_pretrained(MODEL_ID).to(DEVICE)

model.eval()

# -----------------------------
# 3️⃣ Load LVIS labels
# -----------------------------
with open(LVIS_LABELS_PATH, "r") as f:
    lvis_labels = [line.strip() for line in f if line.strip()]

print(f"Loaded {len(lvis_labels)} LVIS labels")

# -----------------------------
# 4️⃣ Load image
# -----------------------------
image = Image.open(IMAGE_PATH).convert("RGB")

# -----------------------------
# 5️⃣ Chunked inference
# -----------------------------
def run_grounding_dino_chunked(image, labels):
    all_boxes = []
    all_scores = []
    all_text_labels = []

    for i in range(0, len(labels), CHUNK_SIZE):
        chunk = labels[i:i + CHUNK_SIZE]
        prompt = ". ".join(label.lower() for label in chunk) + "."

        print(f"Running chunk {i} → {i + len(chunk)}")

        inputs = processor(images=image, text=prompt, return_tensors="pt").to(DEVICE)

        with torch.no_grad():
            outputs = model(**inputs)

        results = processor.post_process_grounded_object_detection(
            outputs=outputs,
            input_ids=inputs.input_ids,
            target_sizes=[image.size[::-1]],
            threshold=BOX_THRESHOLD,
        )[0]

        if len(results["boxes"]) == 0:
            continue

        all_boxes.append(results["boxes"])
        all_scores.append(results["scores"])
        all_text_labels.extend(results["text_labels"])

    if not all_boxes:
        return [], [], []

    all_boxes = torch.cat(all_boxes)
    all_scores = torch.cat(all_scores)

    return all_boxes, all_scores, all_text_labels


# -----------------------------
# 6️⃣ Run detection
# -----------------------------
print("Running detection...")

boxes, scores, text_labels = run_grounding_dino_chunked(image, lvis_labels)

if len(boxes) == 0:
    print("No detections found.")
    exit()

# -----------------------------
# 7️⃣ Cross-chunk NMS
# -----------------------------
keep = ops.nms(boxes, scores, NMS_IOU_THRESHOLD)

boxes = boxes[keep]
scores = scores[keep]
text_labels = [text_labels[i] for i in keep]

# -----------------------------
# 8️⃣ Print results
# -----------------------------
print("\n=== FINAL DETECTIONS ===")

for box, score, label in zip(boxes, scores, text_labels):
    if score >= BOX_THRESHOLD:
        print(
            f"{label:20} "
            f"score={score:.3f} "
            f"box={box.tolist()}"
        )

C:\Users\berin\PycharmProjects\COCO_CNN_App\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading model...


The image processor of type `GroundingDinoImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
Loading weights: 100%|██████████| 1206/1206 [00:02<00:00, 511.06it/s, Materializing param=model.text_projection.weight]                                                                           


Loaded 1203 LVIS labels
Running detection...
Running chunk 0 → 50
Running chunk 50 → 100
Running chunk 100 → 150
Running chunk 150 → 200
Running chunk 200 → 250
Running chunk 250 → 300
Running chunk 300 → 350
Running chunk 350 → 400
Running chunk 400 → 450
Running chunk 450 → 500
Running chunk 500 → 550
Running chunk 550 → 600
Running chunk 600 → 650
Running chunk 650 → 700
Running chunk 700 → 750
Running chunk 750 → 800
Running chunk 800 → 850
Running chunk 850 → 900
Running chunk 900 → 950
Running chunk 950 → 1000
Running chunk 1000 → 1050
Running chunk 1050 → 1100
Running chunk 1100 → 1150
Running chunk 1150 → 1200
Running chunk 1200 → 1203

=== FINAL DETECTIONS ===
laptop _ computer    score=0.483 box=[189.45452880859375, 829.6570434570312, 768.955810546875, 1329.845703125]
table                score=0.331 box=[-4.197925567626953, 878.91259765625, 821.31689453125, 1936.059326171875]
headset              score=0.285 box=[2.222780227661133, 1188.3778076171875, 302.552001953125, 1384.